# 01 - Data Preparation & Feature Engineering

In this notebook, we'll:
1. Load the preprocessed dataset from HuggingFace
2. Apply enhanced feature engineering
3. Generate text embeddings
4. Save the enhanced dataset for model training

In [ ]:
# Imports
import sys
sys.path.append('../')

import os
from dotenv import load_dotenv
from huggingface_hub import login
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.notebook import tqdm

from src.items import Item
from src.features import FeatureEngineer
from src.embeddings import EmbeddingGenerator, batch_embed_items

load_dotenv(override=True)

# Login to HuggingFace
hf_token = os.environ.get('HF_TOKEN')
if hf_token:
    login(hf_token, add_to_git_credential=True)

## Step 1: Load Dataset

We'll use Ed's lite dataset for faster experimentation

In [ ]:
# Load the preprocessed dataset
username = "ed-donner"
dataset_name = f"{username}/items_lite"

print(f"Loading dataset: {dataset_name}")
train, val, test = Item.from_hub(dataset_name)

print(f"\nDataset loaded successfully!")
print(f"Training items: {len(train):,}")
print(f"Validation items: {len(val):,}")
print(f"Test items: {len(test):,}")
print(f"Total items: {len(train) + len(val) + len(test):,}")

In [ ]:
# Inspect a sample item
print("Sample item:")
print(train[0])
print(f"\nSummary: {train[0].summary}")
print(f"Price: ${train[0].price}")

## Step 2: Enhanced Feature Engineering

Extract additional features from the product data

In [ ]:
# Apply enhancements to all items
print("Enhancing training items...")
for item in tqdm(train):
    item.enhance()

print("\nEnhancing validation items...")
for item in tqdm(val):
    item.enhance()

print("\nEnhancing test items...")
for item in tqdm(test):
    item.enhance()

print("\nFeature engineering complete!")

In [ ]:
# Inspect enhanced features
sample = train[0]
print("Enhanced features:")
print(f"Brand: {sample.brand}")
print(f"Word count: {sample.word_count}")
print(f"Character count: {sample.char_count}")
print(f"Has dimensions: {sample.has_dimensions}")
print(f"Is premium: {sample.is_premium}")
print(f"Text quality score: {sample.text_quality_score:.3f}")
print(f"Price bucket: {sample.price_bucket}")
print(f"Price per weight: {sample.price_per_weight}")

## Step 3: Visualize Enhanced Features

In [ ]:
# Price distribution by bucket
from collections import Counter

bucket_counts = Counter([item.price_bucket for item in train])

plt.figure(figsize=(12, 6))
plt.bar(bucket_counts.keys(), bucket_counts.values(), color='skyblue')
plt.title('Price Distribution by Bucket')
plt.xlabel('Price Bucket')
plt.ylabel('Count')
plt.xticks(rotation=45)
for i, (k, v) in enumerate(bucket_counts.items()):
    plt.text(i, v, f"{v:,}", ha='center', va='bottom')
plt.tight_layout()
plt.show()

In [ ]:
# Premium vs non-premium distribution
premium_count = sum(1 for item in train if item.is_premium)
non_premium_count = len(train) - premium_count

plt.figure(figsize=(8, 8))
plt.pie([premium_count, non_premium_count], 
        labels=['Premium', 'Non-Premium'],
        autopct='%1.1f%%',
        colors=['gold', 'lightblue'])
plt.title('Premium vs Non-Premium Products')
plt.show()

## Step 4: Generate Text Embeddings

Use sentence-transformers to create semantic embeddings

In [ ]:
# Generate embeddings for all datasets
print("Generating embeddings...\n")

# Create data directory if it doesn't exist
os.makedirs('../data/embeddings', exist_ok=True)

# Generate and cache embeddings
train_embeddings = batch_embed_items(
    train, 
    model_name='all-MiniLM-L6-v2',
    cache_path='../data/embeddings/train_embeddings.npy'
)

val_embeddings = batch_embed_items(
    val,
    model_name='all-MiniLM-L6-v2', 
    cache_path='../data/embeddings/val_embeddings.npy'
)

test_embeddings = batch_embed_items(
    test,
    model_name='all-MiniLM-L6-v2',
    cache_path='../data/embeddings/test_embeddings.npy'
)

print(f"\nEmbeddings generated!")
print(f"Train embeddings shape: {train_embeddings.shape}")
print(f"Val embeddings shape: {val_embeddings.shape}")
print(f"Test embeddings shape: {test_embeddings.shape}")

In [ ]:
# Store embeddings in items for later use
for item, emb in zip(train, train_embeddings):
    item.embedding = emb.tolist()

for item, emb in zip(val, val_embeddings):
    item.embedding = emb.tolist()

for item, emb in zip(test, test_embeddings):
    item.embedding = emb.tolist()

print("Embeddings stored in items!")

## Step 5: Extract Features for ML Models

In [ ]:
# Initialize feature engineer
feature_engineer = FeatureEngineer()

# Fit and transform training data
train_features = feature_engineer.fit_transform(train)
val_features = feature_engineer.transform(val)
test_features = feature_engineer.transform(test)

print("Feature extraction complete!")
print(f"\nTraining features shape: {train_features.shape}")
print(f"Validation features shape: {val_features.shape}")
print(f"Test features shape: {test_features.shape}")
print(f"\nFeature columns: {list(train_features.columns)}")

In [ ]:
# Display feature statistics
train_features.describe()

## Step 6: Save Enhanced Data

In [ ]:
# Save features
train_features.to_csv('../data/train_features.csv', index=False)
val_features.to_csv('../data/val_features.csv', index=False)
test_features.to_csv('../data/test_features.csv', index=False)

# Save prices
train_prices = np.array([item.price for item in train])
val_prices = np.array([item.price for item in val])
test_prices = np.array([item.price for item in test])

np.save('../data/train_prices.npy', train_prices)
np.save('../data/val_prices.npy', val_prices)
np.save('../data/test_prices.npy', test_prices)

print("Data saved successfully!")
print("\nFiles created:")
print("- data/train_features.csv")
print("- data/val_features.csv")
print("- data/test_features.csv")
print("- data/train_prices.npy")
print("- data/val_prices.npy")
print("- data/test_prices.npy")
print("- data/embeddings/train_embeddings.npy")
print("- data/embeddings/val_embeddings.npy")
print("- data/embeddings/test_embeddings.npy")

## Summary

✅ Loaded dataset from HuggingFace  
✅ Applied enhanced feature engineering  
✅ Generated text embeddings  
✅ Extracted features for ML models  
✅ Saved all data for next steps  

**Next:** Notebook 02 - Build RAG System